# EDA — PubMed RCT 20k

This notebook provides a systematic exploratory data analysis (EDA) for the **PubMed RCT 20k** dataset used in this repository.

It covers:
- Dataset overview and split sizes
- Missingness checks
- Sentence classification label distribution
- Text length / token length summaries (to motivate truncation and batching)
- Weak NER label distribution (entity/tag counts)
- Key visualizations + interpretation that motivates preprocessing and model choices

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_project_root(start: Path) -> Path:
    """Find the repository root by walking upward until src/medical_ai_project exists."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / 'src' / 'medical_ai_project').exists():
            return parent
    raise FileNotFoundError('Could not locate project root (expected src/medical_ai_project).')


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

sns.set_theme(style='whitegrid')
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
from medical_ai_project.data.pubmed_rct20k import (
    add_weak_ner_annotations,
    simple_tokenize,
)
from medical_ai_project.utils.config import load_config

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'default_config.json'
config = load_config(str(CONFIG_PATH))
config['paths']['artifacts_root']

In [ ]:
import datasets

ds_name = config['dataset']['name']
dataset = datasets.load_dataset(ds_name)
dataset

In [ ]:
split_sizes = {split: len(dataset[split]) for split in dataset.keys()}
pd.DataFrame([{'split': k, 'rows': v} for k, v in split_sizes.items()]).sort_values('split')

## Missingness analysis

We check for missing or empty values in the main columns used by the pipelines (`text` and `label` by default).

In [ ]:
text_col = config['dataset']['text_column']
label_col = config['dataset'].get('label_column', 'label')

def missingness_summary(ds, cols: list[str]) -> dict:
    """Compute missing/empty counts for each column."""
    out = {}
    for col in cols:
        values = ds[col]
        n = len(values)
        n_none = sum(v is None for v in values)
        n_empty = sum((isinstance(v, str) and v.strip() == '') for v in values)
        out[col] = {'rows': n, 'none': int(n_none), 'empty_str': int(n_empty)}
    return out

rows = []
for split in dataset.keys():
    stats = missingness_summary(dataset[split], [text_col, label_col])
    for col, s in stats.items():
        rows.append({'split': split, 'column': col, **s})
pd.DataFrame(rows)

## Sentence classification label distribution

The classification track uses the dataset’s sentence labels. Here we visualize the class distribution (train/validation/test) to understand imbalance and motivate settings like class weighting for LSTM baselines.

In [ ]:
def label_counts(ds, label_column: str) -> pd.Series:
    """Return label counts as a pandas Series."""
    values = ds[label_column]
    # The HF dataset may store labels as ints (ClassLabel) or as strings.
    if hasattr(ds.features.get(label_column), 'names'):
        names = list(ds.features[label_column].names)
        mapped = [names[int(v)] if not isinstance(v, str) else v for v in values]
    else:
        mapped = [str(v) for v in values]
    return pd.Series(mapped).value_counts().sort_index()

count_frames = []
for split in ['train', 'validation', 'test']:
    vc = label_counts(dataset[split], label_col)
    count_frames.append(pd.DataFrame({'split': split, 'label': vc.index, 'count': vc.values}))
label_df = pd.concat(count_frames, ignore_index=True)
label_df

In [ ]:
plt.figure(figsize=(10, 4))
ax = sns.barplot(data=label_df, x='label', y='count', hue='split')
ax.set_title('Sentence Label Distribution by Split')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Text length / token length summaries

Transformers require a fixed maximum sequence length (e.g., `max_seq_len=128`). We summarize tokenized sentence lengths to motivate truncation and batching choices.

For speed, we compute statistics on a configurable subset (default: first 25k rows per split). You can increase the cap if you want exact full-dataset stats.

In [ ]:
MAX_ROWS_PER_SPLIT = 25_000

def length_stats(ds, text_column: str, max_rows: int) -> pd.DataFrame:
    """Compute token-length stats for a split using the project tokenizer."""
    n = min(len(ds), max_rows)
    texts = ds.select(range(n))[text_column]
    lengths = np.array([len(simple_tokenize(t)) for t in texts], dtype=np.int32)
    return pd.DataFrame({
        'rows_used': [int(n)],
        'min': [int(lengths.min())],
        'p50': [int(np.quantile(lengths, 0.50))],
        'p90': [int(np.quantile(lengths, 0.90))],
        'p95': [int(np.quantile(lengths, 0.95))],
        'p99': [int(np.quantile(lengths, 0.99))],
        'max': [int(lengths.max())],
        'mean': [float(lengths.mean())],
    })

stats_rows = []
all_lengths = {}
for split in ['train', 'validation', 'test']:
    n = min(len(dataset[split]), MAX_ROWS_PER_SPLIT)
    texts = dataset[split].select(range(n))[text_col]
    lengths = np.array([len(simple_tokenize(t)) for t in texts], dtype=np.int32)
    all_lengths[split] = lengths
    row = length_stats(dataset[split], text_col, MAX_ROWS_PER_SPLIT).iloc[0].to_dict()
    row['split'] = split
    stats_rows.append(row)

pd.DataFrame(stats_rows)[['split','rows_used','min','p50','p90','p95','p99','max','mean']]

In [ ]:
plt.figure(figsize=(10, 4))
for split, lengths in all_lengths.items():
    sns.kdeplot(lengths, label=split, fill=False)
plt.title('Token Length Distribution (KDE)')
plt.xlabel('Tokens per sentence (simple_tokenize)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

### Interpretation (lengths)

- If the 95th/99th percentile exceeds the configured `max_seq_len`, truncation is expected and should be monitored.
- Very long-tail distributions suggest either increasing `max_seq_len` (more compute) or ensuring truncation does not remove critical tokens.

In this repo, `max_seq_len=128` is a compute-friendly default for both the LSTM baselines and transformer fine-tuning.

## Weak NER label analysis

The NER track uses **weak supervision**: a small biomedical lexicon is matched against tokens to generate BIO tags.

We explore:
- how frequently entity tags appear,
- which entity types dominate, and
- what proportion of tokens are labeled as non-entity (`O`).

This helps motivate design choices such as class weighting (for the LSTM baseline) and why token-level accuracy can be misleading when `O` dominates.

In [ ]:
from collections import Counter

NER_SAMPLE_ROWS = 10_000
ner_subset = datasets.DatasetDict({
    split: dataset[split].select(range(min(len(dataset[split]), NER_SAMPLE_ROWS)))
    for split in ['train', 'validation', 'test']
})
annotated = add_weak_ner_annotations(config, ner_subset)

tag_counter = Counter()
entity_counter = Counter()

for split in ['train', 'validation', 'test']:
    for tags in annotated[split]['ner_tags']:
        tag_counter.update(tags)
        for tag in tags:
            if tag.startswith('B-'):
                entity_counter[tag[2:]] += 1

tag_df = (
    pd.DataFrame([{'tag': k, 'count': v} for k, v in tag_counter.items()])
    .sort_values('count', ascending=False)
)
entity_df = (
    pd.DataFrame([{'entity': k, 'count': v} for k, v in entity_counter.items()])
    .sort_values('count', ascending=False)
)
tag_df.head(10), entity_df

In [ ]:
plt.figure(figsize=(7, 4))
ax = sns.barplot(data=entity_df, x='entity', y='count')
ax.set_title('Weak NER: Entity Type Frequency (B- tags)')
ax.set_xlabel('Entity type')
ax.set_ylabel('Count (B- occurrences)')
plt.tight_layout()
plt.show()

### Interpretation (weak NER tags)

- A heavy dominance of `O` is typical and can inflate token accuracy, so entity-level span metrics are more informative.
- Entity imbalance can motivate class weighting for LSTM baselines and careful evaluation by entity type (per-entity F1).
- Weak labels are only as good as the lexicon coverage; low counts for an entity type can limit learnability and evaluation stability.